In [ ]:
import pandas as pd
import json
ans = {}
dataset_list = ['banking', 'stackoverflow', 'clinc', 'hwu', 'mcid']
pd_ans = {'dataset': [], 'ratio': [], 'split': [], 'value': []}
for dataset_name in dataset_list:
    ans[dataset_name] = {}
    for split in ['train', 'dev', 'test']:
        for rate in [0.25, 0.5, 0.75]:
            ans[dataset_name][rate] = {}
            label_ratio = pd.read_csv(f'{dataset_name}/label/label_{rate}.list', header=None)[0]
            ratio = 0.1
            df = pd.read_csv(f'{dataset_name}/{split}.tsv', sep='\t')
            sub_df = df[df['label'].isin(label_ratio)]
            if split in ['train', 'dev']:
                labeled_data = pd.read_csv(f'{dataset_name}/labeled_data/{split}_{ratio}.tsv', sep='\t')['text'].tolist()
                sub_df = sub_df[sub_df['text'].isin(labeled_data)]
            ans[dataset_name][rate][split] = len(sub_df)
            ans[dataset_name][split] = len(df)
            if split in ['train', 'dev']:
                pd_ans['dataset'].append(dataset_name)
                pd_ans['split'].append(f'{split}')
                pd_ans['ratio'].append(rate)
                pd_ans['value'].append(len(sub_df))
            pd_ans['dataset'].append(dataset_name)
            pd_ans['split'].append('\\#label')
            pd_ans['ratio'].append(rate)
            pd_ans['value'].append(len(label_ratio))
        pd_ans['dataset'].append(dataset_name)
        pd_ans['split'].append(f'{split}')
        pd_ans['ratio'].append(1)
        pd_ans['value'].append(len(df))
    num_labels = len(pd.read_csv(f'{dataset_name}/label/label.list', header=None)[0])
    pd_ans['dataset'].append(dataset_name)
    pd_ans['split'].append('\\#label')
    pd_ans['ratio'].append(1)
    pd_ans['value'].append(num_labels)
    ans[dataset_name]['num_labels'] = num_labels
json.dump(ans, open('data_statics.json', 'w'), indent=4)
ans


In [ ]:
pd_ans = pd.DataFrame(pd_ans).sort_values(['dataset', 'ratio', 'split'], ascending=[True, True, False]).drop_duplicates()
pivot_ans = pd_ans.pivot(index='dataset', columns=['ratio', 'split'], values=['value'])
pivot_ans = pivot_ans.applymap(lambda x: '{:,}'.format(x))
pivot_ans.to_excel('data_statics.xlsx')
pivot_ans


In [ ]:
data_statics = pivot_ans.to_latex().replace(' ', '').replace('0000', '')
replace_map = {'Classes': '|Classes|', 'train-labeled': '|Train|', 'train': '|Train|', 'eval': '|Validation|', 'test': '|Test|', 'banking': 'BANKING', 'clinc': 'CLINC', 'stackoverflow': 'STACK', 'thucnews': 'CNEWS', 'atis': 'ATIS', 'snips': 'SNIPS', 'ele': 'REVIEWS', 'news': 'NEWSG', '0000': '', '.00': '', 'lrrrrrrr': 'l|p{1.2cm}p{1.2cm}p{1.2cm}p{1.2cm}p{1.2cm}p{1.2cm}p{1.2cm}', 'toprule\n': 'toprule\nDataset'}
for i, v in replace_map.items():
    data_statics = data_statics.replace(i, v)
print(data_statics.replace(' ', ''))


In [ ]:
import pandas as pd
ans = {}
pd_ans = {'dataset': [], 'ratio': [], 'split': [], 'value': []}
for dataset_name in dataset_list:
    ans[dataset_name] = {}
    ans[dataset_name]['Classes'] = len(pd.read_csv(f'{dataset_name}/label/label.list', header=None))
    all_df = []
    for split in ['train', 'dev', 'test']:
        df = pd.read_csv(f'{dataset_name}/{split}.tsv', sep='\t')
        all_df.append(df)
        ans[dataset_name][f'{split}'] = int(len(df))
    all_df = pd.concat(all_df)
    all_df['len'] = all_df['text'].apply(lambda x: len(str(x).split(' ')))
    ans[dataset_name]['Length (Avg.)'] = all_df['len'].mean().round(2)
ans = pd.DataFrame(ans)
ans = ans.T
for col in ans.columns:
    if 'Length' not in col:
        ans[col] = ans[col].apply(lambda x: int(x))
ans


In [ ]:
replace_map = {'Classes': '|Classes|', 'train': '|Train|', 'eval': '|Validation|', 'test': '|Test|', 'banking': 'BANKING', 'clinc': 'CLINC', 'stackoverflow': 'STACK', 'thucnews': 'CNEWS', 'atis': 'ATIS', 'snips': 'SNIPS', 'ele': 'REVIEWS', 'news': 'NEWSG', '0000': '', '.00': '', 'lrrrrrrr': 'l|p{1.2cm}p{1.2cm}p{1.2cm}p{1.2cm}p{1.2cm}p{1.2cm}p{1.2cm}', 'toprule\n': 'toprule\nDataset'}
ans_latex = ans.to_latex()
for i, v in replace_map.items():
    ans_latex = ans_latex.replace(i, v)
print(ans_latex.replace(' ', ''))
